## Parallelization

Create Langchain with silmultaniosly task run togather

In [ ]:
# import your LLM model
from langchain_ollama import ChatOllama
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display,Markdown

In [ ]:
llm = ChatOllama(model='llama3.1:8b')
results = llm.invoke('Hello my name is Hanif')
print(results)

content='Nice to meet you, Hanif! How are you today? Is there something I can help you with or would you like to chat for a bit?' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-11-29T05:14:48.0682765Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2213902800, 'load_duration': 24421400, 'prompt_eval_count': 16, 'prompt_eval_duration': 221496800, 'eval_count': 32, 'eval_duration': 1966894300, 'model_name': 'llama3.1:8b'} id='run--4b13643f-4230-4bc2-af4f-525c0305775a-0' usage_metadata={'input_tokens': 16, 'output_tokens': 32, 'total_tokens': 48}


In [ ]:
class State(TypedDict):
    topic : str
    characters : str
    settings: str
    premises: str
    story_intro : str

In [ ]:
def generate_characters(state:State):
    """ Generate characters based on description """
    msg = llm.invoke(f"Create two characters names and brief traits for story about {state['topic']}")
    return {"characters": msg.content}

def generate_settings(state:State):
    """Generate story settings"""
    msg = llm.invoke(f"Describe a vivid setting for story about {state['topic']}")
    return {"settings": msg.content}

def generate_premises(state:State):
    """Generate story premises"""
    msg = llm.invoke(f"Generate a one sentence plot premise for story about {state['topic']}")
    return {"premises": msg.content}

def combine_elements(state:State):
    """Combine characters, settings, and premises into a story intro"""
    msg = llm.invoke(
        f"Write a short story introduction using the following elements. "
        f"Using the characters: {state['characters']}, "
        f"settings: {state['settings']}, and "
        f"premises: {state['premises']}, "
        f"write a compelling story introduction."
    )
    return {"story_intro": msg.content}


In [ ]:
graph = StateGraph(State)
graph.add_node("characters", generate_characters)
graph.add_node("settings", generate_settings)
graph.add_node("premises", generate_premises)
graph.add_node("combine", combine_elements)

graph.add_edge(START, "characters")
graph.add_edge(START, "settings")
graph.add_edge(START, "premises")
graph.add_edge("characters", "combine")
graph.add_edge("settings", "combine")
graph.add_edge("premises", "combine")
graph.add_edge("combine", END)

compiled_graph = graph.compile()
mermaid_code = compiled_graph.get_graph().draw_mermaid()
with open("prompt_chaining_graph.mmd", "w") as f:
    f.write(mermaid_code)

In [ ]:
# Stream to monitoring graph execution
config = {'configurable': {'thread_id': '2'}}
state={"topic":"Good governance goverment"}
for chunk in compiled_graph.stream(state,config,stream_mode='updates'):
    print(chunk)

{'premises': {'premises': "In a small island nation where corruption has run rampant, a young and ambitious minister named Maya must navigate the treacherous waters of bureaucracy to pass a landmark anti-corruption bill, while simultaneously confronting her own family's dark past and the country's entrenched powers that be."}}
{'settings': {'settings': 'Here\'s a descriptive passage that sets the stage for a story about good governance in government:\n\n**The Sunshine Plaza**\n\nIn the heart of the bustling city, a sprawling complex stood as a testament to effective governance. The Sunshine Plaza was more than just a building – it was a symbol of transparency, accountability, and efficient administration. Nestled amidst lush greenery, its gleaming glass façade reflected the warmth and optimism that characterized the institution within.\n\nInside, the plaza hummed with activity, as government officials, civil servants, and citizens from all walks of life converged to tackle pressing iss